# AHR Sudoku — Working Hybrid Solver Notebook

This notebook packages the **working AHR-lite / HRM-style Sudoku solver** with:

- compact recurrent AHR model
- hybrid exact repair solver
- checkpoint save/load
- playground cells
- visualization and animation
- stress testing against known solutions

## Cell 1 — imports, config, seed, device

In [ ]:
import os
import csv
import json
import time
import random
from dataclasses import dataclass, asdict
from pathlib import Path
from typing import List, Tuple, Optional, Dict
from itertools import chain
from contextlib import nullcontext

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

try:
    from huggingface_hub import hf_hub_download
except ImportError:
    hf_hub_download = None

try:
    from IPython.display import display, HTML
except ImportError:
    display = None
    HTML = None


def seed_everything(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)


@dataclass
class CFG:
    seed: int = 42

    # data
    source_repo: str = "sapientinc/sudoku-extreme"
    local_train_csv: Optional[str] = None
    local_test_csv: Optional[str] = None
    min_difficulty: Optional[int] = None

    # good A100 run, but still manageable
    max_train_samples: int = 200000
    max_val_samples: int = 10000
    max_test_samples: int = 10000

    # synthetic fallback
    synthetic_train: int = 5000
    synthetic_val: int = 500
    synthetic_test: int = 500
    synthetic_clues_min: int = 24
    synthetic_clues_max: int = 36

    # dataloader
    batch_size: int = 192
    num_workers: int = min(8, os.cpu_count() or 8)

    # compact HRM-style model
    hidden_dim: int = 96
    num_heads: int = 4
    queen_period: int = 4
    queen_ff_mult: int = 2
    steps: int = 40
    detach_interval: Optional[int] = 8

    rho: float = 0.35
    noise_scale: float = 0.18
    obs_gain: float = 2.2
    peer_gain: float = 1.25
    tau_clip: float = 12.0
    clue_on: float = 10.0
    clue_off: float = -10.0

    # training
    epochs: int = 500
    lr: float = 2e-4
    weight_decay: float = 1e-4
    grad_clip: float = 1.0
    stable_tail_k: int = 4
    log_every: int = 20

    # solve
    multi_colonies: int = 16

    # performance
    mixed_precision: bool = True
    compile_model: bool = True
    checkpoint_path: str = "ahr_lite_sudoku_best.pt"


cfg = CFG()

seed_everything(cfg.seed)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

if DEVICE.type == "cuda":
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
    torch.backends.cudnn.benchmark = True
    try:
        torch.set_float32_matmul_precision("high")
    except Exception:
        pass


def amp_context():
    if DEVICE.type == "cuda" and cfg.mixed_precision:
        return torch.autocast(device_type="cuda", dtype=torch.bfloat16)
    return nullcontext()


print("Device:", DEVICE)
print(json.dumps(asdict(cfg), indent=2))

## Cell 2 — CSV parsing, streamed sampling, synthetic fallback, dataset class

In [ ]:
def normalize_grid_string(s: str) -> str:
    s = str(s).strip()
    out = []
    for ch in s:
        if ch in ".0":
            out.append("0")
        elif ch in "123456789":
            out.append(ch)
    out = "".join(out)
    if len(out) != 81:
        raise ValueError(f"Expected 81 cells, got {len(out)}")
    return out


def parse_grid(s: str) -> np.ndarray:
    s = normalize_grid_string(s)
    return np.array([int(ch) for ch in s], dtype=np.int64)


def grid_to_string(arr: np.ndarray) -> str:
    arr = np.asarray(arr).reshape(-1)
    return "".join(str(int(x)) for x in arr)


def iter_pairs_from_csv(path: str, min_difficulty: Optional[int] = None):
    with open(path, newline="") as f:
        reader = csv.reader(f)
        first = next(reader, None)
        if first is None:
            return

        header = [str(x).strip().lower() for x in first]

        def find_idx(candidates):
            for c in candidates:
                if c in header:
                    return header.index(c)
            return None

        q_idx = find_idx(["question", "q", "puzzle", "input", "board", "givens"])
        a_idx = find_idx(["answer", "a", "solution", "label", "target", "output"])
        r_idx = find_idx(["rating", "difficulty", "score"])

        if q_idx is not None and a_idx is not None:
            row_iter = reader
            use_header = True
        else:
            row_iter = chain([first], reader)
            use_header = False

        for row in row_iter:
            try:
                if use_header:
                    puzzle = row[q_idx]
                    solution = row[a_idx]
                    rating = row[r_idx] if (r_idx is not None and r_idx < len(row)) else None
                else:
                    if len(row) >= 4:
                        _, puzzle, solution, rating = row[:4]
                    elif len(row) >= 2:
                        puzzle, solution = row[:2]
                        rating = None
                    else:
                        continue

                if min_difficulty is not None and rating not in [None, ""]:
                    try:
                        if int(float(rating)) < min_difficulty:
                            continue
                    except Exception:
                        pass

                yield normalize_grid_string(puzzle), normalize_grid_string(solution)

            except Exception:
                continue


def reservoir_sample_pairs(
    path: str,
    max_samples: int,
    min_difficulty: Optional[int] = None,
    seed: int = 42,
):
    rng = random.Random(seed)
    sample = []
    seen = 0

    for pair in iter_pairs_from_csv(path, min_difficulty=min_difficulty):
        seen += 1
        if len(sample) < max_samples:
            sample.append(pair)
        else:
            j = rng.randrange(seen)
            if j < max_samples:
                sample[j] = pair

    return sample, seen


def load_repo_split_sampled(
    repo_id: str,
    split_name: str,
    max_samples: int,
    min_difficulty: Optional[int],
    seed: int,
):
    if hf_hub_download is None:
        return [], 0

    path = hf_hub_download(
        repo_id=repo_id,
        filename=f"{split_name}.csv",
        repo_type="dataset",
    )
    return reservoir_sample_pairs(
        path=path,
        max_samples=max_samples,
        min_difficulty=min_difficulty,
        seed=seed,
    )


def canonical_solution_board() -> np.ndarray:
    return np.array(
        [[((r * 3 + r // 3 + c) % 9) + 1 for c in range(9)] for r in range(9)],
        dtype=np.int64,
    )


def permute_solution_board(board: np.ndarray) -> np.ndarray:
    board = board.copy()

    digit_perm = np.arange(10)
    digit_perm[1:] = np.random.permutation(np.arange(1, 10))
    board = digit_perm[board]

    bands = np.random.permutation(3)
    row_perm = np.concatenate([b * 3 + np.random.permutation(3) for b in bands])

    stacks = np.random.permutation(3)
    col_perm = np.concatenate([s * 3 + np.random.permutation(3) for s in stacks])

    board = board[row_perm][:, col_perm]

    if np.random.rand() < 0.5:
        board = board.T

    return board


def make_synthetic_pair(clues_min: int = 24, clues_max: int = 36) -> Tuple[str, str]:
    solved = permute_solution_board(canonical_solution_board())
    puzzle = solved.copy().reshape(-1)

    clues = np.random.randint(clues_min, clues_max + 1)
    blank_count = 81 - clues
    blank_idx = np.random.choice(81, size=blank_count, replace=False)
    puzzle[blank_idx] = 0

    return grid_to_string(puzzle), grid_to_string(solved.reshape(-1))


def synthesize_pairs(n: int, clues_min: int, clues_max: int):
    return [make_synthetic_pair(clues_min, clues_max) for _ in range(n)]


def prepare_pairs(cfg: CFG):
    train_needed = cfg.max_train_samples + cfg.max_val_samples

    if cfg.local_train_csv and Path(cfg.local_train_csv).exists():
        train_pool, seen_train = reservoir_sample_pairs(
            cfg.local_train_csv,
            max_samples=train_needed,
            min_difficulty=cfg.min_difficulty,
            seed=cfg.seed,
        )
    else:
        train_pool, seen_train = load_repo_split_sampled(
            repo_id=cfg.source_repo,
            split_name="train",
            max_samples=train_needed,
            min_difficulty=cfg.min_difficulty,
            seed=cfg.seed,
        )

    if cfg.local_test_csv and Path(cfg.local_test_csv).exists():
        test_pairs, seen_test = reservoir_sample_pairs(
            cfg.local_test_csv,
            max_samples=cfg.max_test_samples,
            min_difficulty=cfg.min_difficulty,
            seed=cfg.seed + 1,
        )
    else:
        test_pairs, seen_test = load_repo_split_sampled(
            repo_id=cfg.source_repo,
            split_name="test",
            max_samples=cfg.max_test_samples,
            min_difficulty=cfg.min_difficulty,
            seed=cfg.seed + 1,
        )

    if len(train_pool) == 0 or len(test_pairs) == 0:
        print("Using synthetic fallback dataset.")
        train_pairs = synthesize_pairs(
            cfg.synthetic_train,
            cfg.synthetic_clues_min,
            cfg.synthetic_clues_max,
        )
        val_pairs = synthesize_pairs(
            cfg.synthetic_val,
            cfg.synthetic_clues_min,
            cfg.synthetic_clues_max,
        )
        test_pairs = synthesize_pairs(
            cfg.synthetic_test,
            cfg.synthetic_clues_min,
            cfg.synthetic_clues_max,
        )
        return train_pairs, val_pairs, test_pairs

    rng = np.random.default_rng(cfg.seed)
    rng.shuffle(train_pool)

    val_take = min(cfg.max_val_samples, max(1, len(train_pool) // 10))
    val_pairs = train_pool[:val_take]
    train_pairs = train_pool[val_take: val_take + cfg.max_train_samples]

    print(f"Scanned train rows: {seen_train}")
    print(f"Scanned test rows:  {seen_test}")
    print(f"Sampled train size: {len(train_pairs)}")
    print(f"Sampled val size:   {len(val_pairs)}")
    print(f"Sampled test size:  {len(test_pairs)}")

    return train_pairs, val_pairs, test_pairs


class SudokuDataset(Dataset):
    def __init__(self, pairs: List[Tuple[str, str]]):
        self.x = torch.tensor(np.stack([parse_grid(p) for p, _ in pairs]), dtype=torch.long)
        self.y = torch.tensor(np.stack([parse_grid(s) for _, s in pairs]), dtype=torch.long)

    def __len__(self):
        return self.x.shape[0]

    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

## Cell 3 — peer graph, energy, validation, pretty printer

In [ ]:
def build_peer_graph() -> torch.Tensor:
    adj = torch.zeros(81, 81, dtype=torch.float32)
    for i in range(81):
        r1, c1 = divmod(i, 9)
        b1r, b1c = r1 // 3, c1 // 3
        for j in range(81):
            if i == j:
                continue
            r2, c2 = divmod(j, 9)
            same_row = r1 == r2
            same_col = c1 == c2
            same_box = (b1r == r2 // 3) and (b1c == c2 // 3)
            if same_row or same_col or same_box:
                adj[i, j] = 1.0
    return adj


PEER_GRAPH = build_peer_graph()
PEER_EDGES = torch.triu(PEER_GRAPH, diagonal=1).nonzero(as_tuple=False)

assert int(PEER_GRAPH[40].sum().item()) == 20

print("Peer graph shape:", tuple(PEER_GRAPH.shape))
print("Center cell peers:", int(PEER_GRAPH[40].sum().item()))
print("Unique undirected peer edges:", int(PEER_EDGES.shape[0]))


def pretty_board(arr) -> str:
    arr = np.asarray(arr).reshape(9, 9)
    lines = []
    for r in range(9):
        parts = []
        for c in range(9):
            val = int(arr[r, c])
            parts.append("." if val == 0 else str(val))
            if c in [2, 5]:
                parts.append("|")
        lines.append(" ".join(parts))
        if r in [2, 5]:
            lines.append("-" * 25)
    return "\n".join(lines)


def logits_to_digits(logits: torch.Tensor) -> torch.Tensor:
    return logits.argmax(dim=-1) + 1


def conflict_energy(pred_board: torch.Tensor, puzzle_board: torch.Tensor) -> torch.Tensor:
    if pred_board.ndim == 1:
        pred_board = pred_board.unsqueeze(0)
    if puzzle_board.ndim == 1:
        puzzle_board = puzzle_board.unsqueeze(0)

    edge_idx = PEER_EDGES.to(pred_board.device)
    u = edge_idx[:, 0]
    v = edge_idx[:, 1]

    peer_conflicts = (pred_board[:, u] == pred_board[:, v]).sum(dim=1)
    clue_conflicts = ((puzzle_board > 0) & (pred_board != puzzle_board)).sum(dim=1)

    return peer_conflicts + clue_conflicts


def is_valid_completed_board(board: np.ndarray) -> bool:
    board = np.asarray(board).reshape(9, 9)
    target = set(range(1, 10))

    for r in range(9):
        if set(board[r, :]) != target:
            return False

    for c in range(9):
        if set(board[:, c]) != target:
            return False

    for br in range(0, 9, 3):
        for bc in range(0, 9, 3):
            if set(board[br:br+3, bc:bc+3].reshape(-1)) != target:
                return False

    return True


def tail_stability_mask(logits_traj: torch.Tensor, k: int = 4) -> torch.Tensor:
    tail = logits_to_digits(logits_traj[-k:])
    return (tail == tail[-1:].expand_as(tail)).all(dim=0).all(dim=-1)

## Cell 4 — compact AHRLiteHRM model

In [ ]:
class WorkerLayer(nn.Module):
    def __init__(self, d_model: int):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(2 * d_model + 9, d_model),
            nn.GELU(),
            nn.Linear(d_model, d_model),
        )
        self.norm = nn.LayerNorm(d_model)

    def forward(self, h: torch.Tensor, p: torch.Tensor, adj: torch.Tensor, deg: torch.Tensor):
        peer_h = torch.einsum("ij,bjd->bid", adj, h) / deg.unsqueeze(0)
        peer_p = torch.einsum("ij,bjd->bid", adj, p) / deg.unsqueeze(0)
        out = self.net(torch.cat([h, peer_h, peer_p], dim=-1))
        return self.norm(h + out)


class QueenRegionLayer(nn.Module):
    def __init__(self, d_model: int, num_heads: int, ff_mult: int = 2):
        super().__init__()
        ff_hidden = ff_mult * d_model
        self.attn = nn.MultiheadAttention(d_model, num_heads, batch_first=True)
        self.ff = nn.Sequential(
            nn.Linear(d_model, ff_hidden),
            nn.GELU(),
            nn.Linear(ff_hidden, d_model),
        )
        self.norm1 = nn.LayerNorm(d_model)
        self.norm2 = nn.LayerNorm(d_model)

    def forward(self, region_tokens: torch.Tensor):
        attn_out, _ = self.attn(region_tokens, region_tokens, region_tokens, need_weights=False)
        x = self.norm1(region_tokens + attn_out)
        x = self.norm2(x + self.ff(x))
        return x


class AHRLiteHRM(nn.Module):
    """
    Compact HRM-style AHR:
      - small shared Worker updated every step
      - slower Queen updated every q_period steps
      - Queen reasons over 27 region tokens (rows, cols, boxes)
      - external world-state / pheromone logits tau
    """

    def __init__(
        self,
        adjacency: torch.Tensor,
        hidden_dim: int = 96,
        num_heads: int = 4,
        queen_period: int = 4,
        queen_ff_mult: int = 2,
        rho: float = 0.35,
        noise_scale: float = 0.18,
        obs_gain: float = 2.2,
        peer_gain: float = 1.25,
        tau_clip: float = 12.0,
        clue_on: float = 10.0,
        clue_off: float = -10.0,
    ):
        super().__init__()

        self.hidden_dim = hidden_dim
        self.queen_period = queen_period
        self.rho = rho
        self.noise_scale = noise_scale
        self.obs_gain = obs_gain
        self.peer_gain = peer_gain
        self.tau_clip = tau_clip
        self.clue_on = clue_on
        self.clue_off = clue_off

        self.register_buffer("adj", adjacency.float())
        self.register_buffer("deg", adjacency.float().sum(dim=-1, keepdim=True).clamp_min(1.0))

        row_ids = torch.tensor([i // 9 for i in range(81)], dtype=torch.long)
        col_ids = torch.tensor([i % 9 for i in range(81)], dtype=torch.long)
        box_ids = torch.tensor([3 * ((i // 9) // 3) + ((i % 9) // 3) for i in range(81)], dtype=torch.long)
        self.register_buffer("row_ids", row_ids)
        self.register_buffer("col_ids", col_ids)
        self.register_buffer("box_ids", box_ids)

        self.blank_tau = nn.Parameter(torch.zeros(1, 81, 9))
        self.pos = nn.Parameter(torch.randn(1, 81, hidden_dim) * 0.02)

        self.encoder = nn.Sequential(
            nn.Linear(19, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
        )

        self.worker = WorkerLayer(hidden_dim)
        self.queen = QueenRegionLayer(hidden_dim, num_heads, ff_mult=queen_ff_mult)

        self.global_pool = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.Tanh(),
        )
        self.z_to_cells = nn.Linear(hidden_dim, hidden_dim, bias=False)

        update_mid = max(hidden_dim // 2, 32)
        self.delta_head = nn.Sequential(
            nn.Linear(hidden_dim, update_mid),
            nn.GELU(),
            nn.Linear(update_mid, 9),
        )

    def make_clue_tensors(self, x_digits: torch.Tensor):
        M = (x_digits > 0).float().unsqueeze(-1)
        safe = (x_digits.clamp(min=1) - 1)
        X = F.one_hot(safe, num_classes=9).float() * M

        clue_logits = torch.full_like(X, self.clue_off)
        clue_logits = clue_logits + X * (self.clue_on - self.clue_off)

        return X, M, clue_logits

    def build_region_tokens(self, h: torch.Tensor):
        B, _, D = h.shape
        grid = h.view(B, 9, 9, D)

        row_tokens = grid.mean(dim=2)
        col_tokens = grid.mean(dim=1)
        box_tokens = (
            grid.view(B, 3, 3, 3, 3, D)
            .permute(0, 1, 3, 2, 4, 5)
            .contiguous()
            .view(B, 9, 9, D)
            .mean(dim=2)
        )

        return torch.cat([row_tokens, col_tokens, box_tokens], dim=1)

    def broadcast_region_tokens(self, region_tokens: torch.Tensor):
        row_tok = region_tokens[:, 0:9, :]
        col_tok = region_tokens[:, 9:18, :]
        box_tok = region_tokens[:, 18:27, :]

        row_ctx = row_tok[:, self.row_ids, :]
        col_ctx = col_tok[:, self.col_ids, :]
        box_ctx = box_tok[:, self.box_ids, :]

        return (row_ctx + col_ctx + box_ctx) / 3.0

    def forward(
        self,
        x_digits: torch.Tensor,
        steps: int = 40,
        use_noise: bool = True,
        detach_interval: Optional[int] = None,
    ):
        B = x_digits.shape[0]
        X, M, clue_logits = self.make_clue_tensors(x_digits)

        tau = self.blank_tau.expand(B, -1, -1).clone()
        tau = tau * (1.0 - M) + clue_logits * M

        z = torch.zeros(B, self.hidden_dim, device=x_digits.device)
        queen_context = torch.zeros(B, 81, self.hidden_dim, device=x_digits.device)

        logits_traj = []
        probs_traj = []
        conf_traj = []

        for t in range(steps):
            p = F.softmax(tau, dim=-1)

            h_in = torch.cat([p, X, M], dim=-1)
            h = self.encoder(h_in)
            h = h + self.pos + self.z_to_cells(z).unsqueeze(1)

            h = self.worker(h, p, self.adj, self.deg)

            if t == 0 or (t % self.queen_period == 0):
                region_tokens = self.build_region_tokens(h)
                region_tokens = self.queen(region_tokens)
                queen_context = self.broadcast_region_tokens(region_tokens)

            h_eff = h + queen_context
            delta_tau = self.delta_head(h_eff)

            obs_err = self.obs_gain * (M * (X - p))
            peer_same = torch.einsum("ij,bjd->bid", self.adj, p) / self.deg.unsqueeze(0)

            if use_noise:
                anneal = 1.0 - (t / max(steps - 1, 1))
                scout = torch.randn_like(tau) * (self.noise_scale * anneal)
            else:
                scout = 0.0

            proposal = tau + delta_tau + obs_err - self.peer_gain * peer_same + scout

            tau = torch.clamp(
                (1.0 - self.rho) * tau + self.rho * proposal,
                min=-self.tau_clip,
                max=self.tau_clip,
            )

            tau = tau * (1.0 - M) + clue_logits * M

            z = self.global_pool(h_eff.mean(dim=1))

            logits_traj.append(tau)
            probs_traj.append(F.softmax(tau, dim=-1))
            conf_traj.append(F.softmax(tau, dim=-1).max(dim=-1).values)

            if detach_interval is not None and (t + 1) % detach_interval == 0:
                tau = tau.detach()
                queen_context = queen_context.detach()
                z = z.detach()

        logits_traj = torch.stack(logits_traj, dim=0)
        probs_traj = torch.stack(probs_traj, dim=0)
        conf_traj = torch.stack(conf_traj, dim=0)

        return logits_traj, probs_traj, conf_traj


model = AHRLiteHRM(
    adjacency=PEER_GRAPH.to(DEVICE),
    hidden_dim=cfg.hidden_dim,
    num_heads=cfg.num_heads,
    queen_period=cfg.queen_period,
    queen_ff_mult=cfg.queen_ff_mult,
    rho=cfg.rho,
    noise_scale=cfg.noise_scale,
    obs_gain=cfg.obs_gain,
    peer_gain=cfg.peer_gain,
    tau_clip=cfg.tau_clip,
    clue_on=cfg.clue_on,
    clue_off=cfg.clue_off,
).to(DEVICE)

print(f"Model parameters: {sum(p.numel() for p in model.parameters()):,}")

## Cell 5 — loss and metrics

In [ ]:
def deep_supervision_loss(
    logits_traj: torch.Tensor,
    x_digits: torch.Tensor,
    y_digits: torch.Tensor,
    blank_bonus: float = 0.90,
    peer_penalty: float = 0.25,
    entropy_penalty: float = 0.015,
    stability_penalty: float = 0.05,
):
    T = logits_traj.shape[0]
    targets = y_digits - 1

    step_weights = torch.linspace(
        0.15,
        1.25,
        T,
        device=logits_traj.device,
        dtype=logits_traj.dtype,
    )

    ce = 0.0
    for t in range(T):
        ce = ce + step_weights[t] * F.cross_entropy(
            logits_traj[t].reshape(-1, 9),
            targets.reshape(-1),
        )
    ce = ce / step_weights.sum()

    blank_mask = (x_digits == 0).float()
    final_logp = F.log_softmax(logits_traj[-1], dim=-1)
    picked = final_logp.gather(-1, targets.unsqueeze(-1)).squeeze(-1)
    blank_ce = -(picked * blank_mask).sum() / blank_mask.sum().clamp_min(1.0)

    final_probs = F.softmax(logits_traj[-1], dim=-1)
    adj = PEER_GRAPH.to(logits_traj.device)
    deg = adj.sum(dim=-1, keepdim=True).clamp_min(1.0)
    peer_same = torch.einsum("ij,bjd->bid", adj, final_probs) / deg.unsqueeze(0)
    peer_reg = (final_probs * peer_same).sum(dim=-1).mean()

    entropy = -(final_probs * torch.log(final_probs.clamp_min(1e-8))).sum(dim=-1).mean()

    if T >= 2:
        prev_probs = F.softmax(logits_traj[-2], dim=-1)
        stability = (final_probs - prev_probs).square().mean()
    else:
        stability = torch.zeros((), device=logits_traj.device)

    loss = (
        ce
        + blank_bonus * blank_ce
        + peer_penalty * peer_reg
        + entropy_penalty * entropy
        + stability_penalty * stability
    )

    return loss, {
        "ce": float(ce.detach().item()),
        "blank_ce": float(blank_ce.detach().item()),
        "peer_reg": float(peer_reg.detach().item()),
        "entropy": float(entropy.detach().item()),
        "stability": float(stability.detach().item()),
        "total": float(loss.detach().item()),
    }


@torch.no_grad()
def batch_metrics(logits_traj: torch.Tensor, x_digits: torch.Tensor, y_digits: torch.Tensor, tail_k: int = 4):
    final_pred = logits_to_digits(logits_traj[-1])
    exact = (final_pred == y_digits).all(dim=1)
    stable = tail_stability_mask(logits_traj, k=tail_k)
    stable_exact = exact & stable

    cell_acc = (final_pred == y_digits).float().mean().item()
    exact_acc = exact.float().mean().item()
    stable_acc = stable.float().mean().item()
    stable_exact_acc = stable_exact.float().mean().item()
    mean_energy = conflict_energy(final_pred, x_digits).float().mean().item()

    return {
        "cell_acc": cell_acc,
        "exact_acc": exact_acc,
        "stable_acc": stable_acc,
        "stable_exact_acc": stable_exact_acc,
        "mean_energy": mean_energy,
    }

## Cell 6 — loaders, training, evaluation, checkpointing

In [ ]:
def make_loaders(cfg: CFG):
    train_pairs, val_pairs, test_pairs = prepare_pairs(cfg)

    train_ds = SudokuDataset(train_pairs)
    val_ds = SudokuDataset(val_pairs)
    test_ds = SudokuDataset(test_pairs)

    loader_kwargs = dict(
        batch_size=cfg.batch_size,
        num_workers=cfg.num_workers,
        pin_memory=(DEVICE.type == "cuda"),
        persistent_workers=(cfg.num_workers > 0),
    )

    train_loader = DataLoader(train_ds, shuffle=True, **loader_kwargs)
    val_loader = DataLoader(val_ds, shuffle=False, **loader_kwargs)
    test_loader = DataLoader(test_ds, shuffle=False, **loader_kwargs)

    print(f"train={len(train_ds)} val={len(val_ds)} test={len(test_ds)}")
    return train_loader, val_loader, test_loader, train_pairs, val_pairs, test_pairs


def train_one_epoch(model, loader, optimizer, cfg: CFG):
    model.train()

    running = {
        "loss": 0.0,
        "cell_acc": 0.0,
        "exact_acc": 0.0,
        "stable_acc": 0.0,
        "stable_exact_acc": 0.0,
        "mean_energy": 0.0,
    }
    n_batches = 0

    pbar = tqdm(loader, desc="train", leave=False)

    for step, (x, y) in enumerate(pbar, start=1):
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        optimizer.zero_grad(set_to_none=True)

        with amp_context():
            logits_traj, probs_traj, conf_traj = model(
                x,
                steps=cfg.steps,
                use_noise=True,
                detach_interval=cfg.detach_interval,
            )
            loss, loss_parts = deep_supervision_loss(logits_traj, x, y)

        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), cfg.grad_clip)
        optimizer.step()

        metrics = batch_metrics(logits_traj.detach(), x, y, tail_k=cfg.stable_tail_k)

        running["loss"] += float(loss.item())
        for k in ["cell_acc", "exact_acc", "stable_acc", "stable_exact_acc", "mean_energy"]:
            running[k] += metrics[k]
        n_batches += 1

        if step % cfg.log_every == 0 or step == len(loader):
            pbar.set_postfix(
                loss=f"{loss.item():.4f}",
                exact=f"{metrics['exact_acc']:.3f}",
                stable=f"{metrics['stable_exact_acc']:.3f}",
                energy=f"{metrics['mean_energy']:.2f}",
            )

    return {k: v / max(n_batches, 1) for k, v in running.items()}


@torch.no_grad()
def evaluate(model, loader, cfg: CFG):
    model.eval()

    running = {
        "loss": 0.0,
        "cell_acc": 0.0,
        "exact_acc": 0.0,
        "stable_acc": 0.0,
        "stable_exact_acc": 0.0,
        "mean_energy": 0.0,
    }
    n_batches = 0

    pbar = tqdm(loader, desc="eval", leave=False)

    for x, y in pbar:
        x = x.to(DEVICE, non_blocking=True)
        y = y.to(DEVICE, non_blocking=True)

        with amp_context():
            logits_traj, probs_traj, conf_traj = model(
                x,
                steps=cfg.steps,
                use_noise=False,
                detach_interval=None,
            )
            loss, _ = deep_supervision_loss(logits_traj, x, y)

        metrics = batch_metrics(logits_traj, x, y, tail_k=cfg.stable_tail_k)

        running["loss"] += float(loss.item())
        for k in ["cell_acc", "exact_acc", "stable_acc", "stable_exact_acc", "mean_energy"]:
            running[k] += metrics[k]
        n_batches += 1

    return {k: v / max(n_batches, 1) for k, v in running.items()}


def fit(model, train_loader, val_loader, cfg: CFG):
    adamw_kwargs = dict(
        params=model.parameters(),
        lr=cfg.lr,
        weight_decay=cfg.weight_decay,
    )

    if DEVICE.type == "cuda":
        try:
            adamw_kwargs["fused"] = True
        except Exception:
            pass

    optimizer = torch.optim.AdamW(**adamw_kwargs)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=cfg.epochs)

    history = []
    best_score = -1.0
    best_state = None

    model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model

    for epoch in range(1, cfg.epochs + 1):
        train_metrics = train_one_epoch(model, train_loader, optimizer, cfg)
        val_metrics = evaluate(model, val_loader, cfg)
        scheduler.step()

        row = {
            "epoch": epoch,
            "lr": optimizer.param_groups[0]["lr"],
            **{f"train_{k}": v for k, v in train_metrics.items()},
            **{f"val_{k}": v for k, v in val_metrics.items()},
        }
        history.append(row)

        score = (
            12.0 * val_metrics["stable_exact_acc"]
            + 2.0 * val_metrics["exact_acc"]
            - 0.01 * val_metrics["mean_energy"]
        )

        if score > best_score:
            best_score = score
            best_state = {k: v.detach().cpu().clone() for k, v in model_to_save.state_dict().items()}
            torch.save(
                {
                    "model_state_dict": best_state,
                    "cfg": asdict(cfg),
                    "history": history,
                },
                cfg.checkpoint_path,
            )

        print(
            f"Epoch {epoch:02d} | "
            f"lr {optimizer.param_groups[0]['lr']:.2e} | "
            f"train loss {train_metrics['loss']:.4f} | "
            f"val loss {val_metrics['loss']:.4f} | "
            f"val exact {val_metrics['exact_acc']:.4f} | "
            f"val stable_exact {val_metrics['stable_exact_acc']:.4f} | "
            f"val energy {val_metrics['mean_energy']:.2f}"
        )

    if best_state is not None:
        model_to_save.load_state_dict(best_state)

    return history


@torch.no_grad()
def test_model(model, test_loader, cfg: CFG):
    metrics = evaluate(model, test_loader, cfg)
    print("Test metrics:")
    print(json.dumps(metrics, indent=2))
    return metrics

## Cell 7 — training history plots

In [ ]:
def plot_history(history: List[Dict]):
    epochs = [row["epoch"] for row in history]

    fig, axes = plt.subplots(1, 3, figsize=(16, 4))

    axes[0].plot(epochs, [row["train_loss"] for row in history], label="train")
    axes[0].plot(epochs, [row["val_loss"] for row in history], label="val")
    axes[0].set_title("Loss")
    axes[0].set_xlabel("Epoch")
    axes[0].legend()

    axes[1].plot(epochs, [row["train_exact_acc"] for row in history], label="train exact")
    axes[1].plot(epochs, [row["val_exact_acc"] for row in history], label="val exact")
    axes[1].plot(epochs, [row["val_stable_exact_acc"] for row in history], label="val stable exact")
    axes[1].set_title("Exact Accuracy")
    axes[1].set_xlabel("Epoch")
    axes[1].legend()

    axes[2].plot(epochs, [row["train_mean_energy"] for row in history], label="train")
    axes[2].plot(epochs, [row["val_mean_energy"] for row in history], label="val")
    axes[2].set_title("Conflict Energy")
    axes[2].set_xlabel("Epoch")
    axes[2].legend()

    plt.tight_layout()
    plt.show()

## Cell 8 — build loaders, create model, train, test

In [ ]:
train_loader, val_loader, test_loader, train_pairs, val_pairs, test_pairs = make_loaders(cfg)

seed_everything(cfg.seed)

base_model = AHRLiteHRM(
    adjacency=PEER_GRAPH.to(DEVICE),
    hidden_dim=cfg.hidden_dim,
    num_heads=cfg.num_heads,
    queen_period=cfg.queen_period,
    queen_ff_mult=cfg.queen_ff_mult,
    rho=cfg.rho,
    noise_scale=cfg.noise_scale,
    obs_gain=cfg.obs_gain,
    peer_gain=cfg.peer_gain,
    tau_clip=cfg.tau_clip,
    clue_on=cfg.clue_on,
    clue_off=cfg.clue_off,
).to(DEVICE)

print(f"Fresh model parameters: {sum(p.numel() for p in base_model.parameters()):,}")

train_model = base_model
if cfg.compile_model and DEVICE.type == "cuda":
    try:
        train_model = torch.compile(base_model)
        print("Model compiled with torch.compile.")
    except Exception as e:
        print("torch.compile failed, continuing without compile:", e)

history = fit(train_model, train_loader, val_loader, cfg)
plot_history(history)

model = base_model
test_metrics = test_model(model, test_loader, cfg)

## Cell 9 — hybrid exact-repair solver

In [ ]:
def board_from_puzzle_string(puzzle: str) -> np.ndarray:
    return parse_grid(puzzle).reshape(9, 9).copy()


def logits_to_candidate_order(logits: torch.Tensor) -> np.ndarray:
    order = torch.argsort(logits, dim=-1, descending=True) + 1
    return order.cpu().numpy()


def valid_candidates(board: np.ndarray, r: int, c: int):
    if board[r, c] != 0:
        return []

    used = set(board[r, :]) | set(board[:, c]) | set(
        board[(r // 3) * 3:(r // 3) * 3 + 3, (c // 3) * 3:(c // 3) * 3 + 3].reshape(-1)
    )
    return [d for d in range(1, 10) if d not in used]


def find_mrv_cell(board: np.ndarray):
    best = None
    best_cands = None

    for r in range(9):
        for c in range(9):
            if board[r, c] == 0:
                cands = valid_candidates(board, r, c)
                if len(cands) == 0:
                    return (r, c, [])
                if best is None or len(cands) < len(best_cands):
                    best = (r, c)
                    best_cands = cands
                    if len(best_cands) == 1:
                        return (r, c, best_cands)

    if best is None:
        return None
    return (best[0], best[1], best_cands)


def solve_with_logits_backtracking(board: np.ndarray, logits: torch.Tensor):
    candidate_order = logits_to_candidate_order(logits)

    def recurse():
        pick = find_mrv_cell(board)
        if pick is None:
            return True

        r, c, cands = pick
        if len(cands) == 0:
            return False

        flat_idx = r * 9 + c
        ranked = [d for d in candidate_order[flat_idx] if d in cands]

        for d in ranked:
            board[r, c] = d
            if recurse():
                return True
            board[r, c] = 0

        return False

    ok = recurse()
    return ok, board.copy()


@torch.no_grad()
def solve_puzzle_hybrid(model, puzzle: str, cfg: CFG):
    model.eval()

    x_np = parse_grid(puzzle)
    x = torch.tensor(x_np, dtype=torch.long, device=DEVICE).unsqueeze(0)

    runs = []

    for k in range(cfg.multi_colonies):
        seed_everything(cfg.seed + 10_000 + k)

        logits_traj, probs_traj, conf_traj = model(
            x,
            steps=cfg.steps,
            use_noise=True,
            detach_interval=None,
        )

        avg_logits = logits_traj[-4:].mean(dim=0)[0].detach().cpu()
        raw_pred = logits_to_digits(logits_traj[-1])[0].detach().cpu()

        board = board_from_puzzle_string(puzzle)
        ok, repaired = solve_with_logits_backtracking(board, avg_logits)

        if ok:
            repaired_t = torch.tensor(repaired.reshape(-1), dtype=torch.long)
            energy = int(conflict_energy(repaired_t.unsqueeze(0), x.cpu()).item())
        else:
            repaired_t = raw_pred
            energy = int(conflict_energy(raw_pred.unsqueeze(0), x).item())

        runs.append({
            "colony": k,
            "raw_pred": raw_pred.clone(),
            "pred": repaired_t.clone(),
            "solved_exactly": bool(ok),
            "energy": energy,
            "stable": bool(tail_stability_mask(logits_traj, cfg.stable_tail_k)[0].item()),
            "mean_conf": float(conf_traj[-1, 0].mean().item()),
            "logits_traj": logits_traj.detach().cpu().clone(),
            "conf_traj": conf_traj.detach().cpu().clone(),
        })

    runs = sorted(
        runs,
        key=lambda r: (
            r["energy"],
            -int(r["solved_exactly"]),
            -int(r["stable"]),
            -r["mean_conf"],
        ),
    )
    return runs[0], runs


def show_solution_report(puzzle: str, result: Dict, target: Optional[str] = None):
    puzzle_arr = parse_grid(puzzle).reshape(9, 9)
    pred_arr = result["pred"].numpy().reshape(9, 9)

    print("Puzzle:")
    print(pretty_board(puzzle_arr))
    print("\nPrediction:")
    print(pretty_board(pred_arr))
    print(f"\nEnergy: {result['energy']}")
    print(f"Stable tail: {result['stable']}")
    print(f"Mean confidence: {result['mean_conf']:.4f}")
    print(f"Hybrid exact repair used: {result['solved_exactly']}")
    print(f"Valid completed Sudoku: {is_valid_completed_board(pred_arr)}")

    if target is not None:
        target_arr = parse_grid(target).reshape(9, 9)
        exact = np.all(pred_arr.reshape(-1) == target_arr.reshape(-1))
        print(f"Exact match to target: {exact}")

## Cell 10 — static visualization

In [ ]:
def draw_board(ax, board, givens=None, confidence=None, title=""):
    board = np.asarray(board).reshape(9, 9)

    if confidence is None:
        confidence = np.ones((9, 9), dtype=np.float32)
    else:
        confidence = np.asarray(confidence).reshape(9, 9)

    ax.imshow(confidence, vmin=0.0, vmax=1.0, cmap="viridis")

    for r in range(10):
        lw = 2.5 if r % 3 == 0 else 0.7
        ax.axhline(r - 0.5, color="white", linewidth=lw)
        ax.axvline(r - 0.5, color="white", linewidth=lw)

    for r in range(9):
        for c in range(9):
            val = int(board[r, c])
            txt = "." if val == 0 else str(val)

            if givens is not None and int(givens[r, c]) != 0:
                color = "white"
                weight = "bold"
            else:
                color = "black" if confidence[r, c] > 0.75 else "white"
                weight = "normal"

            ax.text(c, r, txt, ha="center", va="center", fontsize=12, color=color, fontweight=weight)

    ax.set_xticks([])
    ax.set_yticks([])
    ax.set_title(title)


@torch.no_grad()
def visualize_reasoning_snapshots(model, puzzle: str, steps: int = 40, frames=(0, 13, 26, 39)):
    model.eval()

    x = torch.tensor(parse_grid(puzzle), dtype=torch.long, device=DEVICE).unsqueeze(0)
    logits_traj, probs_traj, conf_traj = model(
        x,
        steps=steps,
        use_noise=False,
        detach_interval=None,
    )

    preds = logits_to_digits(logits_traj[:, 0]).cpu().numpy()
    confs = conf_traj[:, 0].cpu().numpy()
    givens = x[0].cpu().numpy().reshape(9, 9)

    fig, axes = plt.subplots(2, len(frames), figsize=(4 * len(frames), 8))

    for j, t in enumerate(frames):
        t = max(0, min(int(t), steps - 1))

        draw_board(
            axes[0, j],
            preds[t].reshape(9, 9),
            givens=givens,
            confidence=confs[t].reshape(9, 9),
            title=f"Prediction @ step {t + 1}",
        )

        axes[1, j].imshow(confs[t].reshape(9, 9), vmin=0.0, vmax=1.0, cmap="magma")
        for r in range(10):
            lw = 2.5 if r % 3 == 0 else 0.7
            axes[1, j].axhline(r - 0.5, color="white", linewidth=lw)
            axes[1, j].axvline(r - 0.5, color="white", linewidth=lw)
        axes[1, j].set_xticks([])
        axes[1, j].set_yticks([])
        axes[1, j].set_title(f"Confidence @ step {t + 1}")

    plt.tight_layout()
    plt.show()

## Cell 11 — sample puzzle solve + visualization

In [ ]:
sample_idx = 0
sample_puzzle, sample_solution = test_pairs[sample_idx]

print("Sample puzzle:")
print(pretty_board(parse_grid(sample_puzzle)))

print("\nGround truth:")
print(pretty_board(parse_grid(sample_solution)))

best, all_runs = solve_puzzle_hybrid(model, sample_puzzle, cfg)
show_solution_report(sample_puzzle, best, target=sample_solution)

visualize_reasoning_snapshots(
    model,
    sample_puzzle,
    steps=cfg.steps,
    frames=(0, cfg.steps // 3, 2 * cfg.steps // 3, cfg.steps - 1),
)

## Cell 12 — optional reload cell

In [ ]:
ckpt = torch.load(cfg.checkpoint_path, map_location=DEVICE)

reloaded_model = AHRLiteHRM(
    adjacency=PEER_GRAPH.to(DEVICE),
    hidden_dim=cfg.hidden_dim,
    num_heads=cfg.num_heads,
    queen_period=cfg.queen_period,
    queen_ff_mult=cfg.queen_ff_mult,
    rho=cfg.rho,
    noise_scale=cfg.noise_scale,
    obs_gain=cfg.obs_gain,
    peer_gain=cfg.peer_gain,
    tau_clip=cfg.tau_clip,
    clue_on=cfg.clue_on,
    clue_off=cfg.clue_off,
).to(DEVICE)

reloaded_model.load_state_dict(ckpt["model_state_dict"])
reloaded_model.eval()

print("Checkpoint loaded.")

## Cell 13 — manual save / load helpers

In [ ]:
def save_current_model(model, path: str = "ahr_lite_sudoku_manual_save.pt", history=None, extra=None):
    model_to_save = model._orig_mod if hasattr(model, "_orig_mod") else model

    payload = {
        "model_state_dict": model_to_save.state_dict(),
        "cfg": asdict(cfg),
        "history": history,
        "extra": extra,
    }
    torch.save(payload, path)
    print(f"Saved model to: {path}")


def load_saved_model(path: str, device=DEVICE):
    ckpt = torch.load(path, map_location=device)

    loaded_model = AHRLiteHRM(
        adjacency=PEER_GRAPH.to(device),
        hidden_dim=cfg.hidden_dim,
        num_heads=cfg.num_heads,
        queen_period=cfg.queen_period,
        queen_ff_mult=cfg.queen_ff_mult,
        rho=cfg.rho,
        noise_scale=cfg.noise_scale,
        obs_gain=cfg.obs_gain,
        peer_gain=cfg.peer_gain,
        tau_clip=cfg.tau_clip,
        clue_on=cfg.clue_on,
        clue_off=cfg.clue_off,
    ).to(device)

    loaded_model.load_state_dict(ckpt["model_state_dict"])
    loaded_model.eval()

    print(f"Loaded model from: {path}")
    return loaded_model, ckpt

## Cell 14 — optional manual save

In [ ]:
save_current_model(
    model,
    path="ahr_lite_sudoku_final_manual.pt",
    history=history if "history" in globals() else None,
    extra={"note": "manual save after training"}
)

## Cell 15 — playground puzzle parser

In [ ]:
def parse_playground_puzzle(puzzle_text: str) -> str:
    return normalize_grid_string(puzzle_text)


def show_playground_puzzle(puzzle_text: str):
    puzzle = parse_playground_puzzle(puzzle_text)
    print("Parsed puzzle:")
    print(pretty_board(parse_grid(puzzle)))
    return puzzle

## Cell 16 — playground solve cell

In [ ]:
puzzle_text = """
. . . | . 9 . | . . 4
. . . | . 5 . | . 2 .
8 . . | . . 2 | 1 . .
-------------------------
1 . . | 6 . 3 | 9 . .
. . 6 | . 7 . | . . .
9 . . | . 2 1 | 3 . 6
-------------------------
. 9 1 | . . . | 2 . .
. 8 . | . . 9 | . 3 .
3 . . | 2 . . | . . .
"""

play_puzzle = show_playground_puzzle(puzzle_text)

best, all_runs = solve_puzzle_hybrid(model, play_puzzle, cfg)
show_solution_report(play_puzzle, best)

print("\nTop colony energies:")
for r in all_runs[:5]:
    print(
        f"colony={r['colony']:2d} | "
        f"energy={r['energy']:2d} | "
        f"stable={r['stable']} | "
        f"exact_repair={r['solved_exactly']} | "
        f"mean_conf={r['mean_conf']:.4f}"
    )

## Cell 17 — automatic snapshot frames

In [ ]:
def auto_frames(total_steps: int, n_frames: int = 6):
    if n_frames <= 1:
        return (0,)
    frames = np.linspace(0, total_steps - 1, n_frames).round().astype(int)
    frames = sorted(set(frames.tolist()))
    return tuple(frames)


frames = auto_frames(cfg.steps, n_frames=6)

visualize_reasoning_snapshots(
    model,
    play_puzzle,
    steps=cfg.steps,
    frames=frames,
)

## Cell 18 — animation of reasoning

In [ ]:
import matplotlib.animation as animation

@torch.no_grad()
def animate_reasoning(model, puzzle: str, steps: int = 40, use_noise: bool = False):
    model.eval()

    x = torch.tensor(parse_grid(puzzle), dtype=torch.long, device=DEVICE).unsqueeze(0)
    logits_traj, probs_traj, conf_traj = model(
        x,
        steps=steps,
        use_noise=use_noise,
        detach_interval=None,
    )

    preds = logits_to_digits(logits_traj[:, 0]).cpu().numpy()
    confs = conf_traj[:, 0].cpu().numpy()
    givens = x[0].cpu().numpy().reshape(9, 9)

    fig, axes = plt.subplots(1, 2, figsize=(10, 5))

    def draw_frame(t):
        for ax in axes:
            ax.clear()

        draw_board(
            axes[0],
            preds[t].reshape(9, 9),
            givens=givens,
            confidence=confs[t].reshape(9, 9),
            title=f"Prediction @ step {t + 1}/{steps}",
        )

        axes[1].imshow(confs[t].reshape(9, 9), vmin=0.0, vmax=1.0, cmap="magma")
        for r in range(10):
            lw = 2.5 if r % 3 == 0 else 0.7
            axes[1].axhline(r - 0.5, color="white", linewidth=lw)
            axes[1].axvline(r - 0.5, color="white", linewidth=lw)
        axes[1].set_xticks([])
        axes[1].set_yticks([])
        axes[1].set_title(f"Confidence @ step {t + 1}/{steps}")

    anim = animation.FuncAnimation(
        fig,
        draw_frame,
        frames=steps,
        interval=350,
        repeat=True,
    )

    plt.close(fig)

    if HTML is not None:
        return HTML(anim.to_jshtml())
    return anim

## Cell 19 — run the animation

In [ ]:
anim_html = animate_reasoning(
    model,
    play_puzzle,
    steps=cfg.steps,
    use_noise=False,
)

if display is not None:
    display(anim_html)
else:
    print("Animation created, but display() is unavailable in this environment.")

## Cell 20 — inspect raw step-by-step boards as text

In [ ]:
@torch.no_grad()
def print_reasoning_steps(model, puzzle: str, steps: int = 10, chosen_steps=None):
    model.eval()

    x = torch.tensor(parse_grid(puzzle), dtype=torch.long, device=DEVICE).unsqueeze(0)
    logits_traj, probs_traj, conf_traj = model(
        x,
        steps=steps,
        use_noise=False,
        detach_interval=None,
    )

    preds = logits_to_digits(logits_traj[:, 0]).cpu().numpy()

    if chosen_steps is None:
        chosen_steps = auto_frames(steps, n_frames=min(6, steps))

    for t in chosen_steps:
        print(f"\n=== Step {t + 1}/{steps} ===")
        print(pretty_board(preds[t]))


print_reasoning_steps(model, play_puzzle, steps=cfg.steps)

## Cell 21 — single-puzzle evaluation helpers

In [ ]:
@torch.no_grad()
def solve_puzzle_raw(model, puzzle: str, cfg: CFG):
    model.eval()

    x_np = parse_grid(puzzle)
    x = torch.tensor(x_np, dtype=torch.long, device=DEVICE).unsqueeze(0)

    logits_traj, probs_traj, conf_traj = model(
        x,
        steps=cfg.steps,
        use_noise=False,
        detach_interval=None,
    )

    pred = logits_to_digits(logits_traj[-1])[0].detach().cpu()
    stable = bool(tail_stability_mask(logits_traj, cfg.stable_tail_k)[0].item())
    mean_conf = float(conf_traj[-1, 0].mean().item())
    energy = int(conflict_energy(pred.unsqueeze(0), x.cpu()).item())

    return {
        "pred": pred,
        "energy": energy,
        "stable": stable,
        "mean_conf": mean_conf,
        "logits_traj": logits_traj.detach().cpu(),
        "conf_traj": conf_traj.detach().cpu(),
    }


def compare_prediction_to_target(pred: torch.Tensor, target: str):
    target_arr = parse_grid(target)
    pred_arr = pred.numpy().reshape(-1)

    exact = bool(np.all(pred_arr == target_arr))
    valid = bool(is_valid_completed_board(pred_arr.reshape(9, 9)))

    return {
        "exact_match": exact,
        "valid_board": valid,
    }


@torch.no_grad()
def evaluate_one_puzzle(model, puzzle: str, target: str, cfg: CFG, run_hybrid: bool = True):
    row = {}

    t0 = time.perf_counter()
    raw = solve_puzzle_raw(model, puzzle, cfg)
    t1 = time.perf_counter()

    raw_cmp = compare_prediction_to_target(raw["pred"], target)

    row.update({
        "raw_exact": raw_cmp["exact_match"],
        "raw_valid": raw_cmp["valid_board"],
        "raw_energy": raw["energy"],
        "raw_stable": raw["stable"],
        "raw_mean_conf": raw["mean_conf"],
        "raw_time_s": t1 - t0,
    })

    if run_hybrid:
        t2 = time.perf_counter()
        hybrid_best, hybrid_runs = solve_puzzle_hybrid(model, puzzle, cfg)
        t3 = time.perf_counter()

        hybrid_cmp = compare_prediction_to_target(hybrid_best["pred"], target)

        row.update({
            "hybrid_exact": hybrid_cmp["exact_match"],
            "hybrid_valid": hybrid_cmp["valid_board"],
            "hybrid_energy": hybrid_best["energy"],
            "hybrid_stable": hybrid_best["stable"],
            "hybrid_mean_conf": hybrid_best["mean_conf"],
            "hybrid_used_exact_repair": hybrid_best["solved_exactly"],
            "hybrid_time_s": t3 - t2,
        })
    else:
        row.update({
            "hybrid_exact": None,
            "hybrid_valid": None,
            "hybrid_energy": None,
            "hybrid_stable": None,
            "hybrid_mean_conf": None,
            "hybrid_used_exact_repair": None,
            "hybrid_time_s": None,
        })

    return row

## Cell 22 — stress test runner

In [ ]:
def stress_test(
    model,
    pairs: List[Tuple[str, str]],
    cfg: CFG,
    limit: Optional[int] = None,
    run_hybrid: bool = True,
):
    if limit is None:
        eval_pairs = pairs
    else:
        eval_pairs = pairs[:limit]

    records = []

    pbar = tqdm(eval_pairs, desc="stress-test")
    for idx, (puzzle, target) in enumerate(pbar):
        row = evaluate_one_puzzle(model, puzzle, target, cfg, run_hybrid=run_hybrid)
        row["index"] = idx
        row["puzzle"] = puzzle
        row["target"] = target
        records.append(row)

        df_live = pd.DataFrame(records)
        raw_exact_rate = df_live["raw_exact"].mean() if len(df_live) > 0 else 0.0

        if run_hybrid and df_live["hybrid_exact"].notna().any():
            hybrid_exact_rate = df_live["hybrid_exact"].mean()
            pbar.set_postfix(
                raw_exact=f"{raw_exact_rate:.3f}",
                hybrid_exact=f"{hybrid_exact_rate:.3f}",
            )
        else:
            pbar.set_postfix(raw_exact=f"{raw_exact_rate:.3f}")

    df = pd.DataFrame(records)
    return df

## Cell 23 — summary table

In [ ]:
def summarize_stress_test(df: pd.DataFrame):
    summary = {}

    summary["num_puzzles"] = len(df)

    summary["raw_exact_rate"] = float(df["raw_exact"].mean())
    summary["raw_valid_rate"] = float(df["raw_valid"].mean())
    summary["raw_mean_energy"] = float(df["raw_energy"].mean())
    summary["raw_mean_conf"] = float(df["raw_mean_conf"].mean())
    summary["raw_mean_time_s"] = float(df["raw_time_s"].mean())

    if "hybrid_exact" in df.columns and df["hybrid_exact"].notna().any():
        summary["hybrid_exact_rate"] = float(df["hybrid_exact"].mean())
        summary["hybrid_valid_rate"] = float(df["hybrid_valid"].mean())
        summary["hybrid_mean_energy"] = float(df["hybrid_energy"].mean())
        summary["hybrid_mean_conf"] = float(df["hybrid_mean_conf"].mean())
        summary["hybrid_mean_time_s"] = float(df["hybrid_time_s"].mean())
        summary["hybrid_exact_repair_rate"] = float(df["hybrid_used_exact_repair"].mean())

    summary_df = pd.DataFrame([summary])

    try:
        display(summary_df)
    except NameError:
        print(summary_df)

    return summary_df

## Cell 24 — run a stress test on part of the test set

In [ ]:
stress_df = stress_test(
    model=model,
    pairs=test_pairs,
    cfg=cfg,
    limit=250,
    run_hybrid=True,
)

stress_summary = summarize_stress_test(stress_df)

## Cell 25 — save results to CSV

In [ ]:
stress_df.to_csv("ahr_stress_test_results.csv", index=False)
stress_summary.to_csv("ahr_stress_test_summary.csv", index=False)

print("Saved:")
print("- ahr_stress_test_results.csv")
print("- ahr_stress_test_summary.csv")

## Cell 26 — inspect failures

In [ ]:
raw_failures = stress_df[~stress_df["raw_exact"]].copy()
print("Raw failures:", len(raw_failures))
display(raw_failures.head(10))

if stress_df["hybrid_exact"].notna().any():
    hybrid_failures = stress_df[~stress_df["hybrid_exact"]].copy()
    print("Hybrid failures:", len(hybrid_failures))
    display(hybrid_failures.head(10))

## Cell 27 — quick plots

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))

exact_names = ["raw"]
exact_vals = [stress_df["raw_exact"].mean()]

if stress_df["hybrid_exact"].notna().any():
    exact_names.append("hybrid")
    exact_vals.append(stress_df["hybrid_exact"].mean())

axes[0].bar(exact_names, exact_vals)
axes[0].set_ylim(0, 1)
axes[0].set_title("Exact Match Rate")

valid_names = ["raw"]
valid_vals = [stress_df["raw_valid"].mean()]

if stress_df["hybrid_valid"].notna().any():
    valid_names.append("hybrid")
    valid_vals.append(stress_df["hybrid_valid"].mean())

axes[1].bar(valid_names, valid_vals)
axes[1].set_ylim(0, 1)
axes[1].set_title("Valid Board Rate")

time_names = ["raw"]
time_vals = [stress_df["raw_time_s"].mean()]

if stress_df["hybrid_time_s"].notna().any():
    time_names.append("hybrid")
    time_vals.append(stress_df["hybrid_time_s"].mean())

axes[2].bar(time_names, time_vals)
axes[2].set_title("Mean Time Per Puzzle (s)")

plt.tight_layout()
plt.show()

## Cell 28 — clean benchmark report

In [ ]:
summary = {
    "num_puzzles": len(stress_df),
    "raw_exact_rate": stress_df["raw_exact"].mean(),
    "raw_valid_rate": stress_df["raw_valid"].mean(),
    "raw_mean_energy": stress_df["raw_energy"].mean(),
    "raw_mean_conf": stress_df["raw_mean_conf"].mean(),
    "raw_mean_time_s": stress_df["raw_time_s"].mean(),
    "hybrid_exact_rate": stress_df["hybrid_exact"].mean(),
    "hybrid_valid_rate": stress_df["hybrid_valid"].mean(),
    "hybrid_mean_energy": stress_df["hybrid_energy"].mean(),
    "hybrid_mean_conf": stress_df["hybrid_mean_conf"].mean(),
    "hybrid_mean_time_s": stress_df["hybrid_time_s"].mean(),
    "hybrid_exact_repair_rate": stress_df["hybrid_used_exact_repair"].mean(),
}

for k, v in summary.items():
    print(f"{k}: {v}")